In [1]:
print("Akish")

Akish


In [4]:
%pip install apache-airflow

  Using cached markdown_it_py-3.0.0-py3-none-any.whl.metadata (6.9 kB)
  Using cached MarkupSafe-2.1.5-cp38-cp38-win_amd64.whl.metadata (3.1 kB)
  Using cached pathspec-0.12.1-py3-none-any.whl.metadata (21 kB)
  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished with status 'done'
  Using cached termcolor-2.4.0-py3-none-any.whl.metadata (6.1 kB)
  Using cached werkzeug-2.3.8-py3-none-any.whl.metadata (4.1 kB)
  Using cached jsonschema_specifications-2023.12.1-py3-none-any.whl.metadata (3.0 kB)
   ---------------------------------------- 0.0/13.4 MB ? eta -:--:--
   --------------- ------------------------ 5.2/13.4 MB 24.5 MB/s eta 0:00:01
   ------------------------------------ --- 12.3/13.4 MB 30.8 MB/s eta 0:00:01
   ---------------------------------------- 13.4/13.4 MB 27.2 MB/s eta 0:00:00
   ---------------------------------------- 0.0/2.2 MB ? eta -:--:--
   ---------------------------------------- 2.2/2.2 MB 31.1 MB/s eta 0:00:00
   ----------------

In [5]:
import os
import io
import pandas as pd
import pickle
import logging
from airflow.utils.log.logging_mixin import LoggingMixin

c:\Users\akish\anaconda3\envs\venv\lib\site-packages\airflow\__init__.py:36: RuntimeWarning: Airflow currently can be run on POSIX-compliant Operating Systems. For development, it is regularly tested on fairly modern Linux Distros and recent versions of macOS. On Windows you can run it via WSL2 (Windows Subsystem for Linux 2) or via Linux Containers. The work to add Windows support is tracked via https://github.com/apache/airflow/issues/10388, but it is not a high priority.
  warnings.warn(
OSError while attempting to symlink the latest log directory


In [9]:
__file__ = "C:/Users/akish/Bank_Marketing_Prediction_MLOps/dags/src/LoadData.py"
print("abspath",os.path.abspath(__file__))
print("dirname",os.path.dirname(os.path.abspath(__file__)))
PROJECT_DIR = os.path.dirname(os.path.dirname(os.path.dirname(os.path.abspath(__file__))))
DATA_DIR = os.path.join(PROJECT_DIR, "data", "processed")
OUTPUT_FILE_PATH = os.path.join(DATA_DIR, "raw_data.csv")
PICKLE_FILE_PATH = os.path.join(DATA_DIR, "raw_data.pkl")
print("PROJECT_DIR",PROJECT_DIR)
print("DATA_DIR",DATA_DIR)
print("OUTPUT_FILE_PATH",OUTPUT_FILE_PATH)
print("PICKLE_FILE_PATH",PICKLE_FILE_PATH)

abspath C:\Users\akish\Bank_Marketing_Prediction_MLOps\dags\src\LoadData.py
dirname C:\Users\akish\Bank_Marketing_Prediction_MLOps\dags\src
PROJECT_DIR C:\Users\akish\Bank_Marketing_Prediction_MLOps
DATA_DIR C:\Users\akish\Bank_Marketing_Prediction_MLOps\data\processed
OUTPUT_FILE_PATH C:\Users\akish\Bank_Marketing_Prediction_MLOps\data\processed\raw_data.csv
PICKLE_FILE_PATH C:\Users\akish\Bank_Marketing_Prediction_MLOps\data\processed\raw_data.pkl


In [6]:
# Set up project directories and file paths
__file__ = "C:/Users/akish/Bank_Marketing_Prediction_MLOps/dags/src/LoadData.py"
PROJECT_DIR = os.path.dirname(os.path.dirname(os.path.dirname(os.path.abspath(__file__))))
DATA_DIR = os.path.join(PROJECT_DIR, "data", "processed")
OUTPUT_FILE_PATH = os.path.join(DATA_DIR, "raw_data.csv")
PICKLE_FILE_PATH = os.path.join(DATA_DIR, "raw_data.pkl")
LOG_DIR = os.path.join(os.path.dirname(os.path.dirname(os.path.abspath(__file__))), "logs")
LOG_FILE_PATH = os.path.join(LOG_DIR, 'load_data.log')

# Ensure necessary directories exist
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(LOG_DIR, exist_ok=True)

# Set up custom file logger
file_logger = logging.getLogger('file_logger')
file_logger.setLevel(logging.INFO)
file_handler = logging.FileHandler(LOG_FILE_PATH, mode='a')
file_formatter = logging.Formatter('%(asctime)s - %(levelname)s - %(message)s')
file_handler.setFormatter(file_formatter)
file_logger.addHandler(file_handler)

# Set up Airflow logger
airflow_logger = LoggingMixin().log

def custom_log(message, level=logging.INFO):
    """Log to both Airflow and custom file logger"""
    if level == logging.INFO:
        airflow_logger.info(message)
        file_logger.info(message)
    elif level == logging.ERROR:
        airflow_logger.error(message)
        file_logger.error(message)
    elif level == logging.WARNING:
        airflow_logger.warning(message)
        file_logger.warning(message)

def load_data(pickled_file_path=PICKLE_FILE_PATH):
    """
    Load data from a pickle file, convert it to a DataFrame, and save as CSV.
    """
    try:
        custom_log("Starting data loading process")
        custom_log(f"Using pickle file: {pickled_file_path}")

        # Load the pickle file
        with open(pickled_file_path, 'rb') as f:
            loaded_data = pickle.load(f)
        custom_log("Pickle file loaded successfully")

        # Convert the loaded data to a DataFrame and store it
        if isinstance(loaded_data, bytes):
            csv_file = io.StringIO(loaded_data.decode('utf-8'))
            df = pd.read_csv(csv_file, sep=';')
            
            # Save as CSV
            df.to_csv(OUTPUT_FILE_PATH, index=False)
            custom_log(f"Data saved as CSV to: {OUTPUT_FILE_PATH}")
            
            return OUTPUT_FILE_PATH
        else:
            custom_log("Loaded data is not in bytes format as expected", level=logging.ERROR)
            return False

    except FileNotFoundError:
        custom_log(f"Pickle file not found: {pickled_file_path}", level=logging.ERROR)
        return False
    except pd.errors.EmptyDataError:
        custom_log("The CSV data in the pickle file is empty", level=logging.ERROR)
        return False
    except Exception as e:
        custom_log(f"An unexpected error occurred: {e}", level=logging.ERROR)
        return False

In [7]:
result = load_data()
if result:
    custom_log(f"Data loading process completed successfully. Output file: {result}")
else:
    custom_log("Data loading process failed", level=logging.ERROR)

[2025-01-23T11:19:11.077+0530] {3408635377.py:28} INFO - Starting data loading process
[2025-01-23T11:19:11.092+0530] {3408635377.py:29} INFO - Starting data loading process
[2025-01-23T11:19:11.096+0530] {3408635377.py:28} INFO - Using pickle file: C:\Users\akish\Bank_Marketing_Prediction_MLOps\data\processed\raw_data.pkl
[2025-01-23T11:19:11.097+0530] {3408635377.py:29} INFO - Using pickle file: C:\Users\akish\Bank_Marketing_Prediction_MLOps\data\processed\raw_data.pkl
[2025-01-23T11:19:11.125+0530] {3408635377.py:28} INFO - Pickle file loaded successfully
[2025-01-23T11:19:11.127+0530] {3408635377.py:29} INFO - Pickle file loaded successfully
[2025-01-23T11:19:11.581+0530] {3408635377.py:28} INFO - Data saved as CSV to: C:\Users\akish\Bank_Marketing_Prediction_MLOps\data\processed\raw_data.csv
[2025-01-23T11:19:11.584+0530] {3408635377.py:29} INFO - Data saved as CSV to: C:\Users\akish\Bank_Marketing_Prediction_MLOps\data\processed\raw_data.csv
[2025-01-23T11:19:11.590+0530] {340863

In [10]:
# Define paths
PROJECT_DIR = os.path.dirname(os.path.dirname(os.path.dirname(os.path.abspath(__file__))))
DATA_DIR = os.path.join(PROJECT_DIR, "data", "processed")
INPUT_FILE_PATH = os.path.join(DATA_DIR, "raw_data.csv")
PICKLE_FILE_PATH = os.path.join(DATA_DIR, "processed_data.pkl")
INFO_CSV_PATH = os.path.join(DATA_DIR, "dataframe_info.csv")
DESCRIPTION_CSV_PATH = os.path.join(DATA_DIR, "dataframe_description.csv")
LOG_DIR = os.path.join(os.path.dirname(os.path.dirname(os.path.abspath(__file__))), "logs")
LOG_FILE_PATH = os.path.join(LOG_DIR, 'process_data.log')

def is_null_or_unknown(x):
    return pd.isnull(x) or (isinstance(x, str) and x.lower() in ['unknown', 'na', 'n/a', ''])

def process_data(input_file_path=INPUT_FILE_PATH):
    """
    Process the input CSV data, perform data cleaning, and save results.
    """
    try:
        custom_log("Starting data processing")
        custom_log(f"Input file: {input_file_path}")

        # Load CSV data
        df = pd.read_csv(input_file_path, sep=',')
        custom_log(f"Data loaded successfully. DataFrame shape: {df.shape}")

        # Save DataFrame info
        buffer = io.StringIO()
        df.info(buf=buffer)
        info_str = buffer.getvalue().strip().split('\n')
        info_df = pd.DataFrame(info_str[2:], columns=['Info'])
        info_df.to_csv(INFO_CSV_PATH, index=False)
        custom_log(f"DataFrame info saved to {INFO_CSV_PATH}")

        # Save DataFrame description
        description_df = df.describe()
        description_df.to_csv(DESCRIPTION_CSV_PATH)
        custom_log(f"DataFrame description saved to {DESCRIPTION_CSV_PATH}")

        # Check for and handle duplicate rows
        duplicate_rows = df.duplicated().sum()
        if duplicate_rows > 0:
            custom_log(f"Found {duplicate_rows} duplicate rows. Dropping duplicates.")
            df = df.drop_duplicates()
        else:
            custom_log("No duplicate rows found.")

        # Handle null and unknown values
        null_unknown_percentage = df.apply(lambda x: x.apply(is_null_or_unknown).mean()) * 100

        # Log columns and their percentage of null or unknown values
        custom_log("Percentage of null or unknown values in each column:")
        for column, percentage in null_unknown_percentage.items():
            custom_log(f"{column}: {percentage:.2f}%")

        features_to_drop = null_unknown_percentage[null_unknown_percentage > 80].index.tolist()

        if features_to_drop:
            df = df.drop(columns=features_to_drop)
            custom_log(f"Dropped features with >80% null or unknown values: {features_to_drop}")
        else:
            custom_log("No features dropped due to null or unknown values")

        # Fill mode values for unknown values in job and education columns
        for column in ['job', 'education']:
            if column in df.columns:
                mode_value = df[column].mode().iloc[0]
                mask = df[column].apply(is_null_or_unknown)
                df.loc[mask, column] = mode_value
                custom_log(f"Filled unknown values in '{column}' column with mode value: {mode_value}")

        # Save processed data as a pickle file
        df.to_pickle(PICKLE_FILE_PATH)
        custom_log(f"Processed data saved as pickle at {PICKLE_FILE_PATH}")
        custom_log("Data processing completed successfully")

        return PICKLE_FILE_PATH
    except FileNotFoundError:
        custom_log(f"Input file not found: {input_file_path}", level=logging.ERROR)
    except pd.errors.EmptyDataError:
        custom_log("The input CSV file is empty", level=logging.ERROR)
    except Exception as e:
        custom_log(f"An unexpected error occurred during data processing: {e}", level=logging.ERROR)

process_data()

[2025-01-23T11:32:01.028+0530] {3408635377.py:28} INFO - Starting data processing
[2025-01-23T11:32:01.044+0530] {3408635377.py:29} INFO - Starting data processing
[2025-01-23T11:32:01.048+0530] {3408635377.py:28} INFO - Input file: C:\Users\akish\Bank_Marketing_Prediction_MLOps\data\processed\raw_data.csv
[2025-01-23T11:32:01.050+0530] {3408635377.py:29} INFO - Input file: C:\Users\akish\Bank_Marketing_Prediction_MLOps\data\processed\raw_data.csv
[2025-01-23T11:32:01.310+0530] {3408635377.py:28} INFO - Data loaded successfully. DataFrame shape: (45211, 17)
[2025-01-23T11:32:01.313+0530] {3408635377.py:29} INFO - Data loaded successfully. DataFrame shape: (45211, 17)
[2025-01-23T11:32:01.485+0530] {3408635377.py:28} INFO - DataFrame info saved to C:\Users\akish\Bank_Marketing_Prediction_MLOps\data\processed\dataframe_info.csv
[2025-01-23T11:32:01.487+0530] {3408635377.py:29} INFO - DataFrame info saved to C:\Users\akish\Bank_Marketing_Prediction_MLOps\data\processed\dataframe_info.csv


'C:\\Users\\akish\\Bank_Marketing_Prediction_MLOps\\data\\processed\\processed_data.pkl'

In [11]:
# Input and output file paths
INPUT_FILE_PATH = os.path.join(DATA_DIR, "processed_data.pkl")  # Input from HandlingNullValues
INFO_CSV_PATH_BEFORE = os.path.join(DATA_DIR, "datatype_info_before.csv")
INFO_CSV_PATH_AFTER = os.path.join(DATA_DIR, "datatype_info_after.csv")
OUTPUT_PICKLE_PATH = os.path.join(DATA_DIR, "datatype_format_processed.pkl")  # New output pickle path
LOG_FILE_PATH = os.path.join(LOG_DIR,"logs", "process_data.log")



# Function to check and handle data types
def handle_data_types(data):
    logging.info("Checking data types of the dataset.")

    # Log and save the data types before conversion
    data_types_before = data.dtypes
    data_types_before.to_csv(INFO_CSV_PATH_BEFORE, header=True)
    logging.info(f"Saved BEFORE conversion data type information to {INFO_CSV_PATH_BEFORE}")

    for column in data.columns:
        if pd.api.types.is_numeric_dtype(data[column]):
            data[column] = pd.to_numeric(data[column], errors='coerce')
        elif data[column].dtype == 'object':
            data[column] = data[column].astype('string').str.strip().str.lower()  # Format strings
        elif pd.api.types.is_datetime64_any_dtype(data[column]):
            data[column] = pd.to_datetime(data[column], errors='coerce')
        else:
            logging.warning(f"Unhandled data type for column: {column}")

    # Log and save the data types after conversion
    data_types_after = data.dtypes
    data_types_after.to_csv(INFO_CSV_PATH_AFTER, header=True)
    logging.info(f"Saved AFTER conversion data type information to {INFO_CSV_PATH_AFTER}")

    return data

# Main processing function
# Main processing function
# Main processing function
def process_datatype(input_file_path=INPUT_FILE_PATH):
    """
    Process the input CSV data, perform data type handling, and save results.
    """
    try:
        logging.info("Starting data processing")
        logging.info(f"Input file: {input_file_path}")

        # Load data based on file extension
        if os.path.exists(input_file_path):
            if input_file_path.endswith('.csv'):
                data = pd.read_csv(input_file_path)
            elif input_file_path.endswith('.pkl'):
                # Attempt to read as pickle and catch potential errors
                try:
                    data = pd.read_pickle(input_file_path)
                except Exception as e:
                    logging.error(f"Failed to read pickle file: {e}")
                    raise
            else:
                logging.error(f"Unsupported file type: {input_file_path}")
                raise ValueError(f"Unsupported file type: {input_file_path}")

            logging.info(f"Loaded data from {input_file_path} with shape {data.shape}")
        else:
            logging.error(f"File {input_file_path} not found.")
            raise FileNotFoundError(f"{input_file_path} not found.")

        # Check and handle data types
        data = handle_data_types(data)

        # Save the processed data as a pickle file
        data.to_pickle(OUTPUT_PICKLE_PATH)
        logging.info(f"Processed data saved as pickle at {OUTPUT_PICKLE_PATH}")

        logging.info("Data processing completed successfully")
        return OUTPUT_PICKLE_PATH  # Return the output file path for further processing

    except Exception as e:
        logging.error(f"Error occurred: {e}")
        raise  # Reraise the exception after logging it

output_path = process_datatype()
logging.info(f"Output file path: {output_path}")

[2025-01-23T11:35:55.765+0530] {2311432243.py:44} INFO - Starting data processing
[2025-01-23T11:35:55.771+0530] {2311432243.py:45} INFO - Input file: C:\Users\akish\Bank_Marketing_Prediction_MLOps\data\processed\processed_data.pkl
[2025-01-23T11:35:55.870+0530] {2311432243.py:62} INFO - Loaded data from C:\Users\akish\Bank_Marketing_Prediction_MLOps\data\processed\processed_data.pkl with shape (45211, 16)
[2025-01-23T11:35:55.872+0530] {2311432243.py:12} INFO - Checking data types of the dataset.
[2025-01-23T11:35:55.886+0530] {2311432243.py:17} INFO - Saved BEFORE conversion data type information to C:\Users\akish\Bank_Marketing_Prediction_MLOps\data\processed\datatype_info_before.csv
[2025-01-23T11:35:56.140+0530] {2311432243.py:32} INFO - Saved AFTER conversion data type information to C:\Users\akish\Bank_Marketing_Prediction_MLOps\data\processed\datatype_info_after.csv
[2025-01-23T11:35:56.366+0530] {2311432243.py:72} INFO - Processed data saved as pickle at C:\Users\akish\Bank_Ma

In [13]:
import numpy as np

OUTPUT_FILE_PATH = os.path.join(DATA_DIR, "outlier_handled_data.pkl")  # New output file path
LOG_FILE_PATH = os.path.join(LOG_DIR,"logs", "outlier_handling.log")


# Function to detect and handle outliers using IQR
def handle_outliers(data, threshold=1.5):
    logging.info("Starting outlier handling.")
    
    data_handled = data.copy()  # Make a copy to avoid modifying the original data
    
    for column in data.select_dtypes(include=[np.number]).columns:  # Only numeric columns
        Q1 = data[column].quantile(0.25)
        Q3 = data[column].quantile(0.75)
        IQR = Q3 - Q1
        lower_bound = Q1 - threshold * IQR
        upper_bound = Q3 + threshold * IQR
        
        # Log the outliers
        outliers = data[(data[column] < lower_bound) | (data[column] > upper_bound)]
        logging.info(f"Outliers in {column}:\n{outliers}")
        
        # Cap or replace outliers
        data_handled[column] = np.where(data[column] < lower_bound, lower_bound, data[column])
        data_handled[column] = np.where(data[column] > upper_bound, upper_bound, data[column])
        logging.info(f"Handled outliers in {column} using IQR method.")
        
    logging.info("Outlier handling completed.")
    return data_handled

# Main processing function
def process_outlier_handling(input_file_path):
    """
    Process the input data to handle outliers and save results.
    """
    try:
        logging.info("Starting outlier handling process")
        logging.info(f"Input file: {input_file_path}")

        # Load the processed data (assuming it is a pickle file)
        if os.path.exists(input_file_path):
            data = pd.read_pickle(input_file_path)  # Read as a pickle file
            logging.info(f"Loaded data from {input_file_path} with shape {data.shape}")
        else:
            logging.error(f"File {input_file_path} not found.")
            raise FileNotFoundError(f"{input_file_path} not found.")

        # Handle outliers using IQR method
        data_handled = handle_outliers(data, threshold=1.5)

        # Save the outlier-handled data to a new pickle file
        data_handled.to_pickle(OUTPUT_FILE_PATH)
        logging.info(f"Saved outlier-handled data to {OUTPUT_FILE_PATH}")

        logging.info("Outlier handling process completed successfully")
        return OUTPUT_FILE_PATH  # Return the output file path for further processing

    except Exception as e:
        logging.error(f"Error occurred: {e}")
        raise  # Reraise the exception after logging it

output_path = process_outlier_handling(os.path.join(DATA_DIR, "datatype_format_processed.pkl"))
logging.info(f"Output file path: {output_path}")

[2025-01-23T12:07:41.782+0530] {2043077075.py:38} INFO - Starting outlier handling process
[2025-01-23T12:07:41.783+0530] {2043077075.py:39} INFO - Input file: C:\Users\akish\Bank_Marketing_Prediction_MLOps\data\processed\datatype_format_processed.pkl
[2025-01-23T12:07:41.867+0530] {2043077075.py:44} INFO - Loaded data from C:\Users\akish\Bank_Marketing_Prediction_MLOps\data\processed\datatype_format_processed.pkl with shape (45211, 16)
[2025-01-23T12:07:41.869+0530] {2043077075.py:9} INFO - Starting outlier handling.
[2025-01-23T12:07:41.952+0530] {2043077075.py:22} INFO - Outliers in age:
       age      job   marital  education default  balance housing loan   
29158   83  retired   married    primary      no      425      no   no  \
29261   75  retired  divorced    primary      no       46      no   no   
29263   75  retired   married    primary      no     3324      no   no   
29322   83  retired   married   tertiary      no     6236      no   no   
29865   75  retired  divorced   

### Task to process data with data type formatting and outlier handling

In [17]:
__file__ = "C:/Users/akish/Bank_Marketing_Prediction_MLOps/dags/src/data_preprocessing/preprocessing_main.py"
PROJECT_DIR = os.path.dirname(os.path.dirname(os.path.dirname(os.path.dirname(os.path.abspath(__file__)))))
DATA_DIR = os.path.join(PROJECT_DIR, "data", "processed")
INPUT_FILE_PATH = os.path.join(DATA_DIR, "processed_data.pkl")

def preprocess_data(input_file_path= INPUT_FILE_PATH):
    """
    Main preprocessing method that handles outliers and formats data types.

    :param input_file_path: Path to the input data file.
    :param output_file_path: Path to save the processed data file.
    """

    # Format data types
    data = process_datatype(input_file_path)

    # Outlier handling
    data = process_outlier_handling(data)

    print(f"Processed data saved to {data}")
    return data

preprocess_data(INPUT_FILE_PATH)

[2025-01-23T12:12:45.838+0530] {2311432243.py:44} INFO - Starting data processing
[2025-01-23T12:12:45.849+0530] {2311432243.py:45} INFO - Input file: C:\Users\akish\Bank_Marketing_Prediction_MLOps\data\processed\processed_data.pkl


[2025-01-23T12:12:45.919+0530] {2311432243.py:62} INFO - Loaded data from C:\Users\akish\Bank_Marketing_Prediction_MLOps\data\processed\processed_data.pkl with shape (45211, 16)
[2025-01-23T12:12:45.920+0530] {2311432243.py:12} INFO - Checking data types of the dataset.
[2025-01-23T12:12:45.941+0530] {2311432243.py:17} INFO - Saved BEFORE conversion data type information to C:\Users\akish\Bank_Marketing_Prediction_MLOps\data\processed\datatype_info_before.csv
[2025-01-23T12:12:46.129+0530] {2311432243.py:32} INFO - Saved AFTER conversion data type information to C:\Users\akish\Bank_Marketing_Prediction_MLOps\data\processed\datatype_info_after.csv
[2025-01-23T12:12:46.271+0530] {2311432243.py:72} INFO - Processed data saved as pickle at C:\Users\akish\Bank_Marketing_Prediction_MLOps\data\processed\datatype_format_processed.pkl
[2025-01-23T12:12:46.274+0530] {2311432243.py:74} INFO - Data processing completed successfully
[2025-01-23T12:12:46.284+0530] {2043077075.py:38} INFO - Starting 

'C:\\Users\\akish\\Bank_Marketing_Prediction_MLOps\\data\\processed\\outlier_handled_data.pkl'

### Task to perform EDA

In [19]:
import matplotlib.pyplot as plt

__file__ = "C:/Users/akish/Bank_Marketing_Prediction_MLOps/dags/src/eda.py"

# Define paths
PROJECT_DIR = os.path.dirname(os.path.dirname(os.path.dirname(os.path.abspath(__file__))))
DATA_DIR = os.path.join(PROJECT_DIR, "data", "processed")
OUTPUT_DIR = os.path.join(DATA_DIR, "eda_plots")
LOG_DIR = os.path.join(PROJECT_DIR,"dags","logs")
LOG_FILE_PATH = os.path.join(LOG_DIR, "eda.log")

def save_plot(fig, filename):
    """Save the current plot to a file and close it."""
    filepath = os.path.join(OUTPUT_DIR, filename)
    fig.savefig(filepath)
    plt.close(fig)
    custom_log(f"Saved plot: {filepath}")

def perform_eda(input_file_path):
    """Perform EDA tasks and save plots."""
    try:
        custom_log("Starting EDA process")
        custom_log(f"Using input file: {input_file_path}")

        # Load the data
        data = pd.read_pickle(input_file_path)
        custom_log(f"Loaded data from {input_file_path} with shape {data.shape}")
        custom_log(f"Data head:\n{data.head().to_string()}")
        
        # Print current column names
        custom_log(f"Current column names: {data.columns.tolist()}")

        # Target values pie chart
        plt.figure(figsize=(8,8))
        data['y'].value_counts().plot(kind='pie', autopct='%1.1f%%', startangle=90)
        plt.title('Deposit Distribution')
        save_plot(plt.gcf(), "deposit_distribution.png")
        
        # Contact method distribution
        contact_dist = data['contact'].value_counts(normalize=True)
        custom_log(f"Contact method distribution:\n{contact_dist}")
        
        # Contact method countplot
        plt.figure(figsize=(10,6))
        data['contact'].value_counts().plot(kind='bar')
        plt.title('Contact Method Distribution')
        plt.xlabel('Contact Method')
        plt.ylabel('Count')
        save_plot(plt.gcf(), "contact_method_distribution.png")
        
        # Housing loan countplot
        plt.figure(figsize=(8,6))
        data.groupby(['housing', 'y']).size().unstack().plot(kind='bar', stacked=True)
        plt.title('Housing Loan Distribution by Deposit')
        plt.xlabel('Housing Loan')
        plt.ylabel('Count')
        plt.legend(title='Deposit', labels=['No', 'Yes'])
        save_plot(plt.gcf(), "housing_loan_distribution.png")
        
        # Personal loan countplot
        plt.figure(figsize=(8,6))
        data.groupby(['loan', 'y']).size().unstack().plot(kind='bar', stacked=True)
        plt.title('Personal Loan Distribution by Deposit')
        plt.xlabel('Personal Loan')
        plt.ylabel('Count')
        plt.legend(title='Deposit', labels=['No', 'Yes'])
        save_plot(plt.gcf(), "personal_loan_distribution.png")
        
        # Default countplot
        plt.figure(figsize=(8,6))
        data.groupby(['default', 'y']).size().unstack().plot(kind='bar', stacked=True)
        plt.title('Default Status Distribution by Deposit')
        plt.xlabel('Default Status')
        plt.ylabel('Count')
        plt.legend(title='Deposit', labels=['No', 'Yes'])
        save_plot(plt.gcf(), "default_status_distribution.png")
        
        # Month distribution
        month_dist = data['month'].value_counts(normalize=True)
        custom_log(f"Month distribution:\n{month_dist}")
        
        # Month countplot
        plt.figure(figsize=(12,6))
        data.groupby(['month', 'y']).size().unstack().plot(kind='bar', stacked=True)
        plt.title('Month Distribution by Deposit')
        plt.xlabel('Month')
        plt.ylabel('Count')
        plt.legend(title='Deposit', labels=['No', 'Yes'])
        plt.xticks(rotation=45)
        save_plot(plt.gcf(), "month_distribution.png")
        
        # Age distribution
        plt.figure(figsize=(10,6))
        plt.hist(data['age'], bins=30, edgecolor='black')
        plt.title('Age Distribution')
        plt.xlabel('Age')
        plt.ylabel('Count')
        save_plot(plt.gcf(), "age_distribution.png")
        
        # Job distribution
        plt.figure(figsize=(12, 6))
        data['job'].value_counts().plot(kind='bar')
        plt.title('Job Distribution')
        plt.xlabel('Job')
        plt.ylabel('Count')
        plt.xticks(rotation=45, ha='right')
        save_plot(plt.gcf(), "job_distribution.png")
        
        # Marital status countplot
        plt.figure(figsize=(8,6))
        data.groupby(['marital', 'y']).size().unstack().plot(kind='bar', stacked=True)
        plt.title('Distribution of Marital Status by Deposit')
        plt.xlabel('Marital Status')
        plt.ylabel('Count')
        plt.legend(title='Deposit', labels=['No', 'Yes'])
        save_plot(plt.gcf(), "marital_status_distribution.png")
        
        # Education countplot
        plt.figure(figsize=(10,6))
        data.groupby(['education', 'y']).size().unstack().plot(kind='bar', stacked=True)
        plt.title('Distribution of Education among Customers')
        plt.xlabel('Education')
        plt.ylabel('Count')
        plt.legend(title='Deposit', labels=['No', 'Yes'])
        plt.xticks(rotation=45)
        save_plot(plt.gcf(), "education_distribution.png")
        
        # Correlation heatmap
        plt.figure(figsize=(12,10))
        numeric_columns = data.select_dtypes(include=[np.number]).columns
        corr = data[numeric_columns].corr()
        plt.imshow(corr, cmap='coolwarm')
        plt.colorbar()
        plt.xticks(range(len(corr.columns)), corr.columns, rotation=90)
        plt.yticks(range(len(corr.columns)), corr.columns)
        plt.title('Correlation Heatmap (Numeric Columns Only)')
        
        # Add correlation values to the heatmap
        for i in range(len(corr.columns)):
            for j in range(len(corr.columns)):
                plt.text(j, i, f"{corr.iloc[i, j]:.2f}", 
                         ha="center", va="center", color="black")
        
        plt.tight_layout()
        save_plot(plt.gcf(), "correlation_heatmap.png")
        
        custom_log("EDA process completed successfully")
    
    except Exception as e:
        custom_log(f"Error occurred during EDA: {e}", level=logging.ERROR)
        raise

INPUT_FILE_PATH = os.path.join(DATA_DIR, "outlier_handled_data.pkl")
perform_eda(INPUT_FILE_PATH)

[2025-01-23T12:58:56.717+0530] {3408635377.py:28} INFO - Starting EDA process
[2025-01-23T12:58:56.721+0530] {3408635377.py:29} INFO - Starting EDA process
[2025-01-23T12:58:56.724+0530] {3408635377.py:28} INFO - Using input file: C:\Users\akish\Bank_Marketing_Prediction_MLOps\data\processed\outlier_handled_data.pkl
[2025-01-23T12:58:56.725+0530] {3408635377.py:29} INFO - Using input file: C:\Users\akish\Bank_Marketing_Prediction_MLOps\data\processed\outlier_handled_data.pkl
[2025-01-23T12:58:56.821+0530] {3408635377.py:28} INFO - Loaded data from C:\Users\akish\Bank_Marketing_Prediction_MLOps\data\processed\outlier_handled_data.pkl with shape (45211, 16)
[2025-01-23T12:58:56.821+0530] {3408635377.py:29} INFO - Loaded data from C:\Users\akish\Bank_Marketing_Prediction_MLOps\data\processed\outlier_handled_data.pkl with shape (45211, 16)
[2025-01-23T12:58:56.858+0530] {3408635377.py:28} INFO - Data head:
    age           job  marital  education default  balance housing loan  contact  da

<Figure size 800x600 with 0 Axes>

<Figure size 800x600 with 0 Axes>

<Figure size 800x600 with 0 Axes>

<Figure size 1200x600 with 0 Axes>

<Figure size 800x600 with 0 Axes>

<Figure size 1000x600 with 0 Axes>

### Task to perform encoding of categorical variables

In [24]:
from sklearn.preprocessing import LabelEncoder
__file__ = "C:/Users/akish/Bank_Marketing_Prediction_MLOps/dags/src/data_preprocessing/encoding.py"
# Define paths
PROJECT_DIR = os.path.dirname(os.path.dirname(os.path.dirname(os.path.dirname(os.path.abspath(__file__)))))
DATA_DIR = os.path.join(PROJECT_DIR, "data", "processed")


#INPUT_FILE_PATH = os.path.join(DATA_DIR, "outlier_handled_data.pkl") # Data after the outliers are handled
OUTPUT_PICKLE_PATH = os.path.join(DATA_DIR, "encoded_data.pkl")  # New output file path after encoding
OUTPUT_CSV_PATH  = os.path.join(DATA_DIR, "encoded_data.csv")  # encoded csv output path


def encode_categorical_variables(input_file_path):
    try:
        custom_log("Starting categorical encoding process")

        # Load data based on file extension
        if os.path.exists(input_file_path):
            if input_file_path.endswith('.csv'):
                df = pd.read_csv(input_file_path)
            elif input_file_path.endswith('.pkl'):
                try:
                    df = pd.read_pickle(input_file_path)
                    custom_log(f"Loaded data from {input_file_path} with shape {df.shape}")
                except Exception as e:
                    logging.error(f"Failed to read pickle file: {e}")
                    raise
            else:
                logging.error(f"Unsupported file type: {input_file_path}")
                raise ValueError(f"Unsupported file type: {input_file_path}")

            logging.info(f"Loaded data from {input_file_path} with shape {df.shape}")
        else:
            logging.error(f"File {input_file_path} not found.")
            raise FileNotFoundError(f"{input_file_path} not found.")

        # Log column datatypes
        custom_log("Column datatypes:")
        for column, dtype in df.dtypes.items():
            custom_log(f"{column}: {dtype}")

        # Identify categorical columns
        categorical_columns = df.select_dtypes(include=['object', 'string', 'category']).columns
        custom_log(f"Identified {len(categorical_columns)} categorical columns: {list(categorical_columns)}")

        # Initialize LabelEncoder
        le = LabelEncoder()

        # Encode categorical variables and save LabelEncoder objects
        for col in categorical_columns:
            le.fit(df[col].astype(str))
            df[col] = le.transform(df[col].astype(str))
            custom_log(f"Encoded column: {col}")
            
            # Save LabelEncoder object
            le_file_path = os.path.join(DATA_DIR, f"{col}_label_encoder.pkl")
            with open(le_file_path, 'wb') as f:
                pickle.dump(le, f)
            custom_log(f"Saved LabelEncoder for {col} to {le_file_path}")

        # Save the encoded data as CSV
        df.to_csv(OUTPUT_CSV_PATH, index=False)
        custom_log(f"Saved encoded data to CSV file: {OUTPUT_CSV_PATH}")

        # Save the encoded data as pickle
        df.to_pickle(OUTPUT_PICKLE_PATH)
        custom_log(f"Saved encoded data to pickle file: {OUTPUT_PICKLE_PATH}")

        custom_log("Categorical encoding process completed successfully")
        return OUTPUT_PICKLE_PATH

    except Exception as e:
        custom_log(f"An error occurred during categorical encoding: {e}", level=logging.ERROR)
        raise

INPUT_FILE_PATH = os.path.join(DATA_DIR, "outlier_handled_data.pkl")
encode_categorical_variables(INPUT_FILE_PATH)

[2025-01-23T13:07:36.878+0530] {3408635377.py:28} INFO - Starting categorical encoding process


[2025-01-23T13:07:36.884+0530] {3408635377.py:29} INFO - Starting categorical encoding process
[2025-01-23T13:07:36.963+0530] {3408635377.py:28} INFO - Loaded data from C:\Users\akish\Bank_Marketing_Prediction_MLOps\data\processed\outlier_handled_data.pkl with shape (45211, 16)
[2025-01-23T13:07:36.963+0530] {3408635377.py:29} INFO - Loaded data from C:\Users\akish\Bank_Marketing_Prediction_MLOps\data\processed\outlier_handled_data.pkl with shape (45211, 16)
[2025-01-23T13:07:36.967+0530] {978338502.py:32} INFO - Loaded data from C:\Users\akish\Bank_Marketing_Prediction_MLOps\data\processed\outlier_handled_data.pkl with shape (45211, 16)
[2025-01-23T13:07:36.967+0530] {3408635377.py:28} INFO - Column datatypes:
[2025-01-23T13:07:36.967+0530] {3408635377.py:29} INFO - Column datatypes:
[2025-01-23T13:07:36.969+0530] {3408635377.py:28} INFO - age: float64
[2025-01-23T13:07:36.969+0530] {3408635377.py:29} INFO - age: float64
[2025-01-23T13:07:36.969+0530] {3408635377.py:28} INFO - job: st

'C:\\Users\\akish\\Bank_Marketing_Prediction_MLOps\\data\\processed\\encoded_data.pkl'